In [ ]:
!rm -rf embedding

In [ ]:
!git clone https://github.com/ZurabDz/embedding.git

In [ ]:
f# %cd embedding
# !git checkout feat/vllm

In [ ]:
%cd /kaggle/working/embedding
!pip install -q -e ./lm -e ./geo_distill

In [ ]:
import os
os.environ["HF_TOKEN"] = ''

In [ ]:
!ls

In [ ]:
# !python -m geo_distill data --n 500000

In [ ]:
# !pip install -q vllm==0.28.0   

In [ ]:
# !python -m geo_distill local-teacher --help

In [ ]:
# !python -m geo_distill local-teacher --push-to ZurabDz/geo-teacher-qwen3-8b-500k --backend vllm

In [ ]:
# !python -m geo_distill local-teacher --push-to ZurabDz/geo-teacher-qwen3-8b-1.5m

In [ ]:
# !python -m geo_distill fetch-teacher --help

In [ ]:
!python -m geo_distill fetch-teacher ZurabDz/geo-teacher-qwen3-8b-500k

In [ ]:
!pip install -q -U "jax[cuda12]>=0.11" "flax>=0.12.7" "optax>=0.2.8" "orbax-checkpoint>=0.12" "datasets>=5.0.0" "tokenizers>=0.23.1" "grain>=0.2.18" "tqdm>=4.69.0"

In [ ]:
!python -m geo_distill train --mlm-checkpoint ZurabDz/ka-mlm --tokenizer ZurabDz/ka-bpe-32k \
        --epochs 30 --batch-size 84 --dropout 0.1 --push-every 1 --push-to ZurabDz/flax-embedding-2

In [ ]:
# import json, numpy as np
# from geo_distill.data import val_split
# from geo_distill.metrics import similarity_agreement

# sentences = json.load(open("/kaggle/working/embedding/artifacts/sentences.json"))
# teacher = np.load("/kaggle/working/embedding/artifacts/teacher_emb.npy").astype(np.float32)
# _, val_idx = val_split(sentences, 0.1, 0)          # same val-frac/seed as training
# t = teacher[val_idx]
# perfect_student = t - teacher[_].mean(axis=0, keepdims=True)   # train-mean centered
# print(similarity_agreement(perfect_student, t))

# Whatever that prints is your true 100% mark. Every number in your training log should be read as a fraction of it.

In [ ]:
# The levers, ranked

# 1. More data. This is the big one by a wide margin. You have 90k training pairs and are running 120 epochs, so each sentence is seen 120 times. Distillation is data-hungry in a way that epoch count cannot substitute for — the student needs to see the teacher's judgment on new text, not the same text repeatedly. Going to 500k–1M sentences would very likely move your final numbers more than everything else on this list combined. The corpus side is cheap (geo-distill data streams from HF); the cost is teacher embedding time, and your shard cache makes that incremental. Diagnostic: if val Spearman flattens or dips while train loss keeps dropping, you're data-limited, and no hyperparameter will fix it.

# 2. The 384 → 3072/4096 bottleneck. MlmStudent is a Linear(384, out_dim) on top of a 384-wide encoder, so every student embedding lives in a 384-dimensional subspace of the teacher's space, no matter how well you train. Whether that costs you anything depends entirely on how fast your teacher's spectrum decays — the script above reports it. If the top 384 directions hold >97% of the variance, ignore this. If they hold 80%, you're leaving real quality on the table, and the fix is cheap: Qwen3-Embedding is Matryoshka-trained, so you can slice the saved array (emb[:, :768], then renormalize) and retrain with out_dim=768 for free — no re-embedding, since train.py:66 derives out_dim from the array's shape.

# 3. Learning rate. MlmSpec.default_lr = 5e-5 is a sensible fine-tuning default, but your projection head is randomly initialized (normal(0.02)), and 5e-5 is slow for a head starting from noise. Two cheap A/Bs: raise --lr to 1e-4 or 2e-4 with --warmup 1000, or give the head a higher LR than the encoder via optax.multi_transform. Your loss curve doesn't look stuck, so I'd call this worth testing rather than clearly wrong.

# 4. --no-center. Now that you initialize from ka-mlm rather than from scratch, the collapse that motivated centering may not happen, and dropping it removes the train/eval geometry mismatch entirely. One run tells you.

# 5. Batch size. 512 is a contrastive-learning default, but your loss is pure per-example regression (--sim-weight defaults to 0), so there are no in-batch negatives to preserve — the batch is doing nothing except averaging gradients. Dropping to 128–256 gives you 2–4× more optimizer steps per pass over the data. On GPU you'll trade some throughput for it, so pair this with the length-bucketing win above.